In [4]:
import pandas as pd
import sqlite3

In [5]:
# Direct CSV read - no upload needed!
df = pd.read_csv(r'C:\Users\Merlinjuliet\Downloads\archive (2).zip', encoding='latin1')

# Create SQL database
conn = sqlite3.connect(':memory:')
df.to_sql('superstore', conn, if_exists='replace', index=False)

print("✅ Done! Total rows:", len(df))
print(df.head())


✅ Done! Total rows: 9994
   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
1       2  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
2       3  CA-2016-138688   6/12/2016   6/16/2016    Second Class    DV-13045   
3       4  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   
4       5  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   

  Postal Code  Region       Product ID         Category Sub-Categor

In [7]:
# a. SELECT, WHERE, ORDER BY, GROUP BY
print("=== a. SELECT, WHERE, ORDER BY ===")
q1 = pd.read_sql_query("""
SELECT `Customer Name`, `Category`, `Sales`, `Region`
FROM superstore
WHERE `Sales` > 500
ORDER BY `Sales` DESC
LIMIT 10
""", conn)
print(q1)

=== a. SELECT, WHERE, ORDER BY ===
        Customer Name         Category      Sales   Region
0         Sean Miller       Technology  22638.480    South
1        Tamara Chand       Technology  17499.950  Central
2        Raymond Buch       Technology  13999.960     West
3        Tom Ashbrook       Technology  11199.968     East
4        Hunter Lopez       Technology  10499.970     East
5       Adrian Barton  Office Supplies   9892.740  Central
6        Sanjit Chand  Office Supplies   9449.950  Central
7        Bill Shonely       Technology   9099.930     East
8        Sanjit Engle       Technology   8749.950    South
9  Christopher Conant       Technology   8399.976     East


In [6]:
print("\n=== a. GROUP BY ===")
q2 = pd.read_sql_query("""
SELECT `Category`, COUNT(*) as Total_Orders,
SUM(`Sales`) as Total_Sales,
AVG(`Sales`) as Avg_Sales
FROM superstore
GROUP BY `Category`
""", conn)
print(q2)



=== a. GROUP BY ===
          Category  Total_Orders  Total_Sales   Avg_Sales
0        Furniture          2121  741999.7953  349.834887
1  Office Supplies          6026  719047.0320  119.324101
2       Technology          1847  836154.0330  452.709276


In [11]:
# b. JOINS
print("\n=== b. INNER JOIN ===")
q3 = pd.read_sql_query("""
SELECT a.`Customer Name`, a.`Category`, 
       b.`Region`, b.`Sales`
FROM superstore a
INNER JOIN superstore b 
ON a.`Customer ID` = b.`Customer ID`
WHERE a.`Category` = 'Furniture'
LIMIT 10
""", conn)
print(q3)



=== b. INNER JOIN ===
  Customer Name   Category   Region    Sales
0   Claire Gute  Furniture  Central    5.344
1   Claire Gute  Furniture  Central   18.160
2   Claire Gute  Furniture  Central  131.376
3   Claire Gute  Furniture    South  261.960
4   Claire Gute  Furniture    South  731.940
5   Claire Gute  Furniture  Central    5.344
6   Claire Gute  Furniture  Central   18.160
7   Claire Gute  Furniture  Central  131.376
8   Claire Gute  Furniture    South  261.960
9   Claire Gute  Furniture    South  731.940


In [15]:
print("\n=== b. LEFT JOIN ===")
q4 = pd.read_sql_query("""
SELECT a.`Customer Name`, a.`Segment`,
       b.`Sales`, b.`Profit`
FROM superstore a
LEFT JOIN superstore b
ON a.`Order ID` = b.`Order ID`
WHERE a.`Segment` = 'Consumer'
LIMIT 10
""", conn)
print(q4)


=== b. LEFT JOIN ===
     Customer Name   Segment     Sales    Profit
0      Claire Gute  Consumer  261.9600   41.9136
1      Claire Gute  Consumer  731.9400  219.5820
2      Claire Gute  Consumer  261.9600   41.9136
3      Claire Gute  Consumer  731.9400  219.5820
4   Sean O'Donnell  Consumer   22.3680    2.5164
5   Sean O'Donnell  Consumer  957.5775 -383.0310
6   Sean O'Donnell  Consumer   22.3680    2.5164
7   Sean O'Donnell  Consumer  957.5775 -383.0310
8  Brosina Hoffman  Consumer    7.2800    1.9656
9  Brosina Hoffman  Consumer   18.5040    5.7825


In [7]:
print("\n=== b. RIGHT JOIN ===")
q5 = pd.read_sql_query("""
SELECT a.`Region`, b.`Customer Name`, 
       b.`Sales`
FROM superstore a
LEFT JOIN superstore b
ON a.`Order ID` = b.`Order ID`
WHERE b.`Sales` > 1000
LIMIT 10
""", conn)
print(q5)


=== b. RIGHT JOIN ===
  Region    Customer Name     Sales
0   West  Brosina Hoffman  1706.184
1   West  Brosina Hoffman  1706.184
2   West  Brosina Hoffman  1706.184
3   West  Brosina Hoffman  1706.184
4   West  Brosina Hoffman  1706.184
5   West  Brosina Hoffman  1706.184
6   West  Brosina Hoffman  1706.184
7   West      Emily Burns  1044.630
8   East  Tracy Blumstein  3083.430
9   East  Tracy Blumstein  3083.430


In [18]:
# c. SUBQUERY
print("\n=== c. SUBQUERY ===")
q6 = pd.read_sql_query("""
SELECT `Customer Name`, `Sales`, `Category`
FROM superstore
WHERE `Sales` > (
    SELECT AVG(`Sales`) FROM superstore
)
ORDER BY `Sales` DESC
LIMIT 10
""", conn)
print(q6)


=== c. SUBQUERY ===
        Customer Name      Sales         Category
0         Sean Miller  22638.480       Technology
1        Tamara Chand  17499.950       Technology
2        Raymond Buch  13999.960       Technology
3        Tom Ashbrook  11199.968       Technology
4        Hunter Lopez  10499.970       Technology
5       Adrian Barton   9892.740  Office Supplies
6        Sanjit Chand   9449.950  Office Supplies
7        Bill Shonely   9099.930       Technology
8        Sanjit Engle   8749.950       Technology
9  Christopher Conant   8399.976       Technology


In [8]:
# d. AGGREGATE FUNCTIONS
print("\n=== d. AGGREGATE FUNCTIONS ===")
q7 = pd.read_sql_query("""
SELECT `Region`,
       SUM(`Sales`) as Total_Sales,
       AVG(`Sales`) as Avg_Sales,
       MAX(`Sales`) as Max_Sale,
       MIN(`Sales`) as Min_Sale,
       COUNT(*) as Total_Orders
FROM superstore
GROUP BY `Region`
""", conn)
print(q7)


=== d. AGGREGATE FUNCTIONS ===
    Region  Total_Sales   Avg_Sales   Max_Sale  Min_Sale  Total_Orders
0  Central  501239.8908  215.772661  17499.950     0.444          2323
1     East  678781.2400  238.336110  11199.968     0.852          2848
2    South  391721.9050  241.803645  22638.480     1.167          1620
3     West  725457.8245  226.493233  13999.960     0.990          3203


In [22]:
# e. VIEW
print("\n=== e. CREATE VIEW ===")
conn.execute("""
CREATE VIEW IF NOT EXISTS high_sales AS
SELECT `Customer Name`, `Category`, 
       `Sales`, `Profit`, `Region`
FROM superstore
WHERE `Sales` > 1000
""")
q8 = pd.read_sql_query("""
SELECT * FROM high_sales 
ORDER BY Sales DESC
LIMIT 10
""", conn)
print(q8)


=== e. CREATE VIEW ===
        Customer Name         Category      Sales     Profit   Region
0         Sean Miller       Technology  22638.480 -1811.0784    South
1        Tamara Chand       Technology  17499.950  8399.9760  Central
2        Raymond Buch       Technology  13999.960  6719.9808     West
3        Tom Ashbrook       Technology  11199.968  3919.9888     East
4        Hunter Lopez       Technology  10499.970  5039.9856     East
5       Adrian Barton  Office Supplies   9892.740  4946.3700  Central
6        Sanjit Chand  Office Supplies   9449.950  4630.4755  Central
7        Bill Shonely       Technology   9099.930  2365.9818     East
8        Sanjit Engle       Technology   8749.950  2799.9840    South
9  Christopher Conant       Technology   8399.976  1119.9968     East


In [9]:
# f. INDEX
print("\n=== f. INDEX ===")
conn.execute("""
CREATE INDEX IF NOT EXISTS 
idx_category ON superstore(`Category`)
""")
conn.execute("""
CREATE INDEX IF NOT EXISTS 
idx_sales ON superstore(`Sales`)
""")
q9 = pd.read_sql_query("""
SELECT `Category`, SUM(`Sales`) as Total
FROM superstore
GROUP BY `Category`
ORDER BY Total DESC
""", conn)
print(q9)

print("\n ALL QUERIES COMPLETED SUCCESSFULLY!")


=== f. INDEX ===
          Category        Total
0       Technology  836154.0330
1        Furniture  741999.7953
2  Office Supplies  719047.0320

 ALL QUERIES COMPLETED SUCCESSFULLY!


In [14]:
print("=== CATEGORY WISE SALES ===") 
q2 = pd.read_sql_query(""" SELECT `Category`, COUNT(*) as Total_Orders, SUM(`Sales`) as Total_Sales, AVG(`Sales`) as Avg_Sales FROM superstore GROUP BY `Category` """, conn)
print(q2)

=== CATEGORY WISE SALES ===
          Category  Total_Orders  Total_Sales   Avg_Sales
0        Furniture          2121  741999.7953  349.834887
1  Office Supplies          6026  719047.0320  119.324101
2       Technology          1847  836154.0330  452.709276
